In [1]:
import os
import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim
import shutil
import random
import gc
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

# Set fixed random seed for reproducibility
random.seed(42)

## Dataset Reorganization Explanation

This code restructures a chest X-ray pneumonia dataset into new train/validation/test splits.

### 1. **Creates New Folder Structure**
```python
/chest_xray_new/
├── train/
│   ├── NORMAL/
│   └── PNEUMONIA/
├── val/
│   ├── NORMAL/
│   └── PNEUMONIA/
└── test/
    ├── NORMAL/
    └── PNEUMONIA/
```
Uses os.makedirs() to create these directories if they don't exist.

### 2. **Combines and Shuffles Data**
- Merges images from all original splits (train/val/test) for each class (NORMAL/PNEUMONIA)  
- Randomizes the file order to eliminate any original ordering bias

### 3. **Splits Data into New Proportions**
- 80% Training (first 80% of shuffled files)  
- 10% Validation (next 10%)  
- 10% Testing (final 10%)

### 4. **Copies Files to New Structure**
- Uses `shutil.copy()` to populate the new directories with redistributed files

### **Why Do This?**
- Creates fresh splits since original dataset had imbalanced distributions  
- Ensures no data leakage between splits  
- Provides standard 80-10-10 split for experimentation  

⚠️ **Important Note**: This overwrites the original splits - don't use if:  
- You need to preserve the original test set as a true hold-out  
- The dataset already has validated splits for reproducibility

## Optimized Dataset Restructuring
This script restructures the chest X-ray pneumonia dataset into train/validation/test splits efficiently using **multiprocessing** for parallel file copying.

### 🔹 Optimizations:
- **Parallel Copying:** Uses all CPU cores for faster execution.
- **Avoids Data Leakage:** Ensures a proper train/validation/test split.
- **Reproducible Splits:** Fixes randomness with `random.seed(42)`.

In [2]:
# ======================================================================
# RESTRUCTURE DATASET (Optimized with Progress Bar)
# ======================================================================

dataset_path = '/kaggle/input/chest-xray-pneumonia/chest_xray'
new_dataset_path = '/kaggle/working/chest_xray_new'

# Delete old directory if exists
if os.path.exists(new_dataset_path):
    shutil.rmtree(new_dataset_path)

# Create new dataset structure
for split in ['train', 'val', 'test']:
    for cls in ['NORMAL', 'PNEUMONIA']:
        os.makedirs(os.path.join(new_dataset_path, split, cls), exist_ok=True)

# Function for parallel file copying
def copy_file(src_dest):
    shutil.copy(*src_dest)

# Process each class
for cls in ['NORMAL', 'PNEUMONIA']:
    all_files = []

    # Collect all files from existing dataset
    for split in ['train', 'val', 'test']:
        source_folder = os.path.join(dataset_path, split, cls)
        if not os.path.exists(source_folder):
            print(f"Warning: {source_folder} does not exist!")
            continue
        files = [os.path.join(source_folder, f) for f in os.listdir(source_folder)]
        all_files.extend(files)

    # Shuffle and split dataset
    random.seed(42)  # Ensure reproducibility
    random.shuffle(all_files)
    train_files = all_files[:int(len(all_files) * 0.8)]
    val_files = all_files[int(len(all_files) * 0.8):int(len(all_files) * 0.9)]
    test_files = all_files[int(len(all_files) * 0.9):]

    # Prepare file copy tasks
    copy_tasks = []
    for src in train_files:
        copy_tasks.append((src, os.path.join(new_dataset_path, 'train', cls, os.path.basename(src))))
    for src in val_files:
        copy_tasks.append((src, os.path.join(new_dataset_path, 'val', cls, os.path.basename(src))))
    for src in test_files:
        copy_tasks.append((src, os.path.join(new_dataset_path, 'test', cls, os.path.basename(src))))

    # Use multiprocessing with tqdm for progress tracking
    with Pool(cpu_count()) as pool:
        list(tqdm(pool.imap(copy_file, copy_tasks), total=len(copy_tasks), desc=f"Copying {cls} Files"))

print("✅ Dataset restructuring complete!")

Copying PNEUMONIA Files: 100%|██████████| 4273/4273 [00:07<00:00, 593.34it/s]

✅ Dataset restructuring complete!


In [3]:
# ======================================================================
# 1. CONFIGURATION
# ======================================================================
SCALE_FACTOR = 2  # For scale; choose 2, 3, or 4
DATASET_PATH = '/kaggle/working/chest_xray_new'  # Your balanced dataset
LR_PATH = '/kaggle/working/chest_xray_LR'  # For downscaled images
BICUBIC_PATH = '/kaggle/working/chest_xray_bicubic'  # Bicubic upscaled results
BATCH_SIZE = 16  # Number of images to process in one batch

# Check if GPU is available and set a flag
use_gpu = cv2.cuda.getCudaEnabledDeviceCount() > 0
print(f"GPU acceleration: {'Enabled' if use_gpu else 'Disabled'}")

GPU acceleration: Disabled


In [4]:
# ======================================================================
# 2. CREATE DIRECTORIES
# ======================================================================
for path in [LR_PATH, BICUBIC_PATH]:
    for split in ['train', 'val', 'test']:
        for cls in ['NORMAL', 'PNEUMONIA']:
            os.makedirs(os.path.join(path, split, cls), exist_ok=True)

In [5]:
# ======================================================================
# 3. PRECOMPUTE IMAGE DIMENSIONS (Optional, comment out if too slow)
# ======================================================================
def precompute_dimensions():
    print("Precomputing image dimensions...")
    image_dimensions = {}
    
    for split in ['train', 'val', 'test']:
        for cls in ['NORMAL', 'PNEUMONIA']:
            src_dir = os.path.join(DATASET_PATH, split, cls)
            if not os.path.exists(src_dir):
                continue
                
            for img_name in tqdm(os.listdir(src_dir), desc=f"Computing dimensions for {split}/{cls}"):
                img_path = os.path.join(src_dir, img_name)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    # Store dimensions adjusted for scale factor
                    h, w = img.shape
                    h = h - (h % SCALE_FACTOR)
                    w = w - (w % SCALE_FACTOR)
                    image_dimensions[img_path] = (h, w)
    
    print(f"Dimensions precomputed for {len(image_dimensions)} images")
    return image_dimensions

# Comment this out if precomputing is too slow for your dataset
# image_dimensions = precompute_dimensions()

In [6]:
# ======================================================================
# 4. OPTIMIZED PROCESSING PIPELINE
# ======================================================================
def process_image_batch(args):
    """Process a batch of images in parallel"""
    batch, split, cls = args
    batch_metrics = {'psnr': [], 'ssim': []}
    
    for img_name in batch:
        orig_img_path = os.path.join(DATASET_PATH, split, cls, img_name)
        lr_img_path = os.path.join(LR_PATH, split, cls, img_name)
        bicubic_img_path = os.path.join(BICUBIC_PATH, split, cls, img_name)
        
        # Get dimensions from cache if available, otherwise compute
        # if 'image_dimensions' in globals() and orig_img_path in image_dimensions:
        #     h, w = image_dimensions[orig_img_path]
        #     img = cv2.imread(orig_img_path, cv2.IMREAD_GRAYSCALE)
        #     if h != img.shape[0] or w != img.shape[1]:
        #         img = cv2.resize(img, (w, h))
        # else:
        # Read image (force grayscale conversion if needed)
        img = cv2.imread(orig_img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f"Warning: Could not read {orig_img_path}")
            continue
            
        # Handle odd dimensions for clean scaling
        h, w = img.shape
        h = h - (h % SCALE_FACTOR)
        w = w - (w % SCALE_FACTOR)
        if h != img.shape[0] or w != img.shape[1]:
            img = cv2.resize(img, (w, h))
        
        # Use GPU acceleration if available
        if use_gpu:
            try:
                # Upload to GPU
                gpu_mat = cv2.cuda_GpuMat()
                gpu_mat.upload(img)
                
                # Downscale with bicubic
                gpu_lr = cv2.cuda.resize(gpu_mat, (w//SCALE_FACTOR, h//SCALE_FACTOR), interpolation=cv2.INTER_CUBIC)
                
                # Upscale back to original size
                gpu_bicubic = cv2.cuda.resize(gpu_lr, (w, h), interpolation=cv2.INTER_CUBIC)
                
                # Download results
                lr_img = gpu_lr.download()
                bicubic_img = gpu_bicubic.download()
            except Exception as e:
                # Fallback to CPU on error
                print(f"GPU error: {e}, falling back to CPU")
                lr_img = cv2.resize(img, (w//SCALE_FACTOR, h//SCALE_FACTOR), interpolation=cv2.INTER_CUBIC)
                bicubic_img = cv2.resize(lr_img, (w, h), interpolation=cv2.INTER_CUBIC)
        else:
            # CPU processing
            lr_img = cv2.resize(img, (w//SCALE_FACTOR, h//SCALE_FACTOR), interpolation=cv2.INTER_CUBIC)
            bicubic_img = cv2.resize(lr_img, (w, h), interpolation=cv2.INTER_CUBIC)
        
        # Compute metrics
        psnr = cv2.PSNR(img, bicubic_img)
        ssim_val = ssim(img, bicubic_img, data_range=255, win_size=11, gaussian_weights=True)
        
        # Save processed images with optimized parameters (faster write, slightly larger files)
        cv2.imwrite(lr_img_path, lr_img, [cv2.IMWRITE_JPEG_QUALITY, 90])
        cv2.imwrite(bicubic_img_path, bicubic_img, [cv2.IMWRITE_JPEG_QUALITY, 90])
        
        batch_metrics['psnr'].append(psnr)
        batch_metrics['ssim'].append(ssim_val)
    
    return batch_metrics

In [7]:
# ======================================================================
# 5. OPTIMIZED MAIN PROCESSING FUNCTION
# ======================================================================
def process_dataset():
    metrics = {'psnr': [], 'ssim': []}
    all_tasks = []
    
    # Prepare batches of images for parallel processing
    for split in ['train', 'val', 'test']:
        for cls in ['NORMAL', 'PNEUMONIA']:
            src_dir = os.path.join(DATASET_PATH, split, cls)
            if not os.path.exists(src_dir):
                continue
                
            img_names = os.listdir(src_dir)
            
            # Create batches
            for i in range(0, len(img_names), BATCH_SIZE):
                batch = img_names[i:i+BATCH_SIZE]
                all_tasks.append((batch, split, cls))
    
    # Clear memory before intensive processing
    gc.collect()
    
    # Use multiprocessing with progress tracking
    num_workers = max(1, cpu_count() - 1)  # Leave one CPU for system
    print(f"Processing {len(all_tasks)} batches with {num_workers} workers...")
    
    with Pool(num_workers) as pool:
        results = list(tqdm(
            pool.imap(process_image_batch, all_tasks),
            total=len(all_tasks),
            desc="Processing Image Batches"
        ))
    
    # Combine results
    for batch_result in results:
        metrics['psnr'].extend(batch_result['psnr'])
        metrics['ssim'].extend(batch_result['ssim'])
    
    # Clear memory after intensive processing
    gc.collect()
    
    return metrics

In [8]:
# ======================================================================
# 6. RUN THE OPTIMIZED PROCESSING
# ======================================================================
metrics = process_dataset()

Processing 368 batches with 3 workers...


Processing Image Batches: 100%|██████████| 368/368 [13:06<00:00,  2.14s/it]


In [9]:
# ======================================================================
# 7. METRIC REPORTING
# ======================================================================
print(f"\n[Baseline Metrics for Scale Factor {SCALE_FACTOR}X]")
print(f"Average PSNR: {np.mean(metrics['psnr']):.2f} dB")
print(f"Average SSIM: {np.mean(metrics['ssim']):.4f}")
print(f"Processed {len(metrics['psnr'])} images")


[Baseline Metrics for Scale Factor 2X]
Average PSNR: 39.66 dB
Average SSIM: 0.9654
Processed 5856 images


In [10]:
# ======================================================================
# 8. DATA VERIFICATION
# ======================================================================
print("\nSample Verification:")
if len(metrics['psnr']) > 0:  # Only show if images processed
    sample_idx = 0
    sample_split = 'train'
    sample_cls = 'NORMAL'
    
    try:
        orig_img_path = os.path.join(DATASET_PATH, sample_split, sample_cls, 
                                     os.listdir(os.path.join(DATASET_PATH, sample_split, sample_cls))[sample_idx])
        lr_img_path = os.path.join(LR_PATH, sample_split, sample_cls, 
                                  os.listdir(os.path.join(LR_PATH, sample_split, sample_cls))[sample_idx])
        
        print(f"Original Image: {orig_img_path}")
        print(f"LR Image Size: {cv2.imread(lr_img_path).shape}")
    except Exception as e:
        print(f"Error during verification: {e}")
else:
    print("No images processed - check directory paths!")


Sample Verification:
Original Image: /kaggle/working/chest_xray_new/train/NORMAL/NORMAL2-IM-0385-0001.jpeg
LR Image Size: (695, 838, 3)


## Compress Directories into a ZIP File

In [11]:
# Define ZIP file name
zip_filename = "/kaggle/working/chest_xray_all.zip"

# Remove old ZIP if it exists
if os.path.exists(zip_filename):
    os.remove(zip_filename)

# Define the base working directory
working_dir = "/kaggle/working"

# List of folders to include in the ZIP archive
folders_to_zip = ["chest_xray_LR", "chest_xray_bicubic", "chest_xray_new"]

# Create a ZIP archive containing all specified folders
shutil.make_archive(zip_filename.replace(".zip", ""), 'zip', working_dir, base_dir=".", logger=None)

print(f"✅ Dataset saved as {zip_filename}")

✅ Dataset saved as /kaggle/working/chest_xray_all.zip
